<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import os

# repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
# repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

# if os.path.exists("../"+repo_name):
#     print("Repository already present, update...")
#     !git pull
# else:
#     print("Repository clone...")
#     !git clone -b rag {repo_url}
#     %cd {repo_name}

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# parquet_path = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection.parquet'

In [4]:
# import importlib
# import rag
# # from rag import load_parquet
# importlib.reload(rag)

# ds = rag.load_parquet(parquet_patth)

In [5]:
# ds['content']

In [6]:
# import phase
!pip install beir rank_bm25 faiss-cpu hnswlib ir_measures

from rank_bm25 import BM25Okapi
import numpy as np
import faiss
import hnswlib
import time
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from datasets import load_dataset


In [8]:
import os
parquet_path = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/subset_collection_it.parquet'
if (os.path.exists(parquet_path)):
  print("Ok")

ds = load_dataset("parquet", data_files=parquet_path, split="train")

# Get total number of documents without loading them
num_docs = len(ds)
corpus_ids = list(range(num_docs))

print(f"Total documents: {num_docs}")

Ok


Generating train split: 0 examples [00:00, ? examples/s]

Total documents: 1048576


In [9]:

bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16},)
#x_enc = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def embed_sparse(docs):
    return list(map(lambda text: text.split(" "), docs))

def embed_dense(encoder, docs):
    return encoder.encode(docs, convert_to_numpy=True, batch_size=4, show_progress_bar=True, normalize_embeddings=True)

def dense_search(query, top_k):
    # get similarity scores
    query_embedding = embed_dense(bi_enc, [query])[:1]
    doc_inds, scores = index_hnsw.knn_query(query_embedding, k=top_k)

    # format to run
    run = {corpus_ids[int(ind)]: float(s) for ind, s in zip(doc_inds[0], scores[0])}
    return run

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [14]:
# create dense index processing dataset in chunks and storing them in the drive at the specified path
output_path="/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings_subset_it"

import os
import pickle

# Assicura che la cartella esista
os.makedirs(output_path, exist_ok=True)

chunk_size = 10000  # Numero di documenti per ogni salvataggio
start_index = 0    # Cambia questo valore se devi ripartire da un punto interrotto

# Sostituiamo len(corpus_docs) con num_docs (che abbiamo definito prima)
for i in range(start_index, num_docs, chunk_size):
    end_index = min(i + chunk_size, num_docs)
    print(f"Processando blocchi da {i} a {end_index}...")

    # CHANGED HERE: Estraiamo la fetta direttamente dal dataset su disco.
    # Questo carica in RAM SOLO i 5000 documenti di questo blocco.
    batch_docs = ds[i:end_index]['content']

    # Genera gli embedding (usa la tua funzione embed_dense con batch_size=4)
    embeddings = embed_dense(bi_enc, batch_docs)

    # Salva il pezzetto su Drive
    file_name = f"embeddings_chunk_{i}_{end_index}.pkl"
    with open(os.path.join(output_path, file_name), 'wb') as f:
        pickle.dump(embeddings, f)

    print(f"Salvato: {file_name}")

Processando blocchi da 0 a 10000...


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Salvato: embeddings_chunk_0_10000.pkl
Processando blocchi da 10000 a 20000...


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Salvato: embeddings_chunk_10000_20000.pkl
Processando blocchi da 20000 a 30000...


...

Salvato: embeddings_chunk_70000_80000.pkl
Processando blocchi da 80000 a 90000...


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Salvato: embeddings_chunk_80000_90000.pkl
Processando blocchi da 90000 a 100000...


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
import os
import pickle
import numpy as np
import re
import hnswlib
import gc

path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings_subset_it"

# 1. Get and sort files
if not os.path.exists(path):
  os.makedirs(path)
files = [f for f in os.listdir(path) if f.endswith('.pkl')]
def extract_number(filename):
    match = re.search(r'chunk_(\d+)_', filename)
    return int(match.group(1)) if match else 0
files.sort(key=extract_number)

# 2. Initialize HNSW Index FIRST
dim = 1024 # BAAI/bge-m3 dimension
index_hnsw = hnswlib.Index(space='cosine', dim=dim)
index_hnsw.init_index(max_elements=num_docs, ef_construction=200, M=32)

# 3. Incrementally load, add to index, and delete from RAM
current_id = 0
for file in files:
    with open(os.path.join(path, file), 'rb') as f:
        chunk_embeddings = pickle.load(f)

    chunk_size = chunk_embeddings.shape[0]
    ids_for_chunk = list(range(current_id, current_id + chunk_size))

    # Add to index
    index_hnsw.add_items(chunk_embeddings, ids_for_chunk)
    print(f"Aggiunto all'indice: {file} (Elementi: {chunk_size})")

    current_id += chunk_size

    # FREE MEMORY explicitly
    del chunk_embeddings
    del ids_for_chunk
    gc.collect()

print("Indice popolato con successo!")

# Initialize the generative component
from transformers import pipeline, AutoTokenizer
pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

Aggiunto all'indice: embeddings_chunk_0_10000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_10000_20000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_20000_30000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_30000_40000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_40000_50000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_50000_60000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_60000_70000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_70000_80000.pkl (Elementi: 10000)
Aggiunto all'indice: embeddings_chunk_80000_90000.pkl (Elementi: 10000)
Indice popolato con successo!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [45]:
# Initialize the generative component
from transformers import pipeline, AutoTokenizer
from huggingface_hub import login

from google.colab import userdata
TOKEN=userdata.get('HF_TOKEN')

import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [17]:
#Save complete embeddings at the specified path
index_path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings/kurdish_wiki_index.bin"
index_hnsw.save_index(index_path)
print("Indice salvato! Ora puoi respirare.")

Indice salvato! Ora puoi respirare.


In [55]:

def rag(query, top_k):
    # create system and user messages
    system_prompt = "You are helpful assistant. Answer the question given the supportive information."
    user_prompt = f"Question: {query}"

    # search relevant documents
    if top_k > 0:
        # runs = rrf_search(query, top_k)
        runs = dense_search(query, top_k)
        docs = [ds[int(doc_id)]['content'] for doc_id in runs]
        docs_context = "\n".join(docs)
        user_prompt += f"\n Referenced knowledge: {docs_context}"

    # format prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # generate the answer
    outputs = pipe(
        prompt,
        max_new_tokens=128,
        do_sample=True,
        max_length=None,
        temperature=0.2,
        eos_token_id=pipe.tokenizer.eos_token_id,
        pad_token_id=pipe.tokenizer.pad_token_id
    )
    predicted_answer = outputs[0]['generated_text'][len(prompt):].strip()

    if top_k > 0:
      return prompt, predicted_answer, docs
    else: return prompt, predicted_answer

In [56]:
  top_k = 0
  prompt, predicted_answer = rag("Chi ha vinto il AEGON Pro Series Edgbaston 2013?", top_k)
  print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 May 2026

You are helpful assistant. Answer the question given the supportive information.<|eot_id|><|start_header_id|>user<|end_header_id|>

Question: Chi ha vinto il AEGON Pro Series Edgbaston 2013?<|eot_id|><|start_header_id|>assistant<|end_header_id|>


Predicted answer: Non ho informazioni su chi ha vinto il torneo AEGON Pro Series Edgbaston 2013.


In [59]:
top_k = 2
prompt, predicted_answer, docs = rag("Chi ha vinto il AEGON Pro Series Edgbaston 2013?", top_k)
i=0
for doc in docs:
  print(f"===============================\nDoc {i}\n:{doc}\n===============================")
  i=i+1
print("==================Answer=============")
print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Doc 0
:<Table>
| AEGON Pro Series Edgbaston 2013 |
| --- | --- |
Sport:  Tennis 
Data: 8 aprile - 14 aprile 
Superficie: Cemento (indoor) 
Località: Edgbaston, Regno Unito 
Campioni
Singolare maschile  Laurynas Grigelis 
Singolare femminile  Ekaterina Byčkova 
Doppio maschile  Luke Bambridge /  George Morgan 
Doppio femminile  Kristina Barrois /  Ana Vrljić 
  2014  
</Table>
LAEGON Pro Series Edgbaston 2013 è stato un torneo di tennis facente parte della categoria ITF Men's Circuit nell'ambito dell'ITF Men's Circuit 2013 e dell'ITF Women's Circuit nell'ambito dell'ITF Women's Circuit 2013. Il torneo femminile si è giocato a Edgbaston nel Regno Unito dall'8 al 14 aprile 2013, quello maschile dal 28 ottobre 2013 al 3 novembre su campi in cemento indoor.
Doc 1
:Il doppio del torneo di tennis AEGON Pro Series Edgbaston 2013, facente parte della categoria ITF Women's Circuit, ha avuto come vincitrici Kristina Barrois e Ana Vrljić che hanno battuto in finale Richèl Hogenkamp e Stephanie Vog

In [20]:
#Debug: test single query vs document
# 1. Prendi il blocco "incriminato" e la tua domanda
target_doc = """<Table>
| 10 cm M>  8 Gebirgshaubitze |
| --- | --- |
M. 10
Tipo: obice da montagna
Origine: Austria-Ungheria
Impiego
Utilizzatori: KuK Armee
Conflitti: prima guerra mondiale
Produzione
Data progettazione: 1908-1910
Date di produzione: 1908-1916
Ritiro dal servizio: 1918
Varianti: 10 cm M. 10 Gebirgshaubitze
Descrizione
Calibro: 104 mm
Peso proiettile: 14,3 kg
Azionamento: otturatore a vite interrotta eccentrico
Elevazione: +43°
voci di armi d'artiglieria presenti su Wikipedia
</Table>
Il 10 cm Gebirgshaubitze M. 8 era un obice da montagna austro-ungarico, impiegato durante la prima guerra mondiale.
"""
query = "Qual è il peso del proiettile 10 cm Gebirgshaubitze M. 8?"

# 2. Genera gli embedding solo per questi due
target_emb = bi_enc.encode(["passage: " + target_doc], normalize_embeddings=True)
query_emb = bi_enc.encode([query], normalize_embeddings=True)

# 3. Calcola la similarità coseno manuale (deve essere alta, es. > 0.7)
similarity = np.dot(query_emb, target_emb.T)
print(f"Similarità tra domanda e blocco specifico: {similarity[0][0]:.4f}")

Similarità tra domanda e blocco specifico: 0.6929
